# ACT Model to OpenVINO IR Conversion

The Action Chunking Transformer (ACT) is a model that learns a generative model over action sequences for bimanual manipulation. See the original paper for details: [Action Chunking Transformer](https://arxiv.org/pdf/2304.13705).

In this tutorial, we show how to convert a Unitree ACT policy (stored in the LeRobot format) to the OpenVINO Intermediate Representation (IR), producing FP32 artifacts.


## Dependency and Core Installation Verification
Run the next cell to install all the required packages.

In [ ]:
# Environment Setup (lerobot + ACT dependencies)
import os, sys, subprocess, shutil, pathlib, textwrap

def vrun(args, msg, check=True, env=None):
    print('[STEP]', msg)
    return subprocess.run([sys.executable] + args, check=check, env=env)

REPO_URL = 'https://github.com/unitreerobotics/unitree_IL_lerobot.git'
REPO_DIR = pathlib.Path('unitree_IL_lerobot')
PARENT_DIR = REPO_DIR / 'unitree_lerobot'      # submodule / inner repo
NESTED_DIR = PARENT_DIR / 'lerobot'            # nested python package

# Desired commits
COMMIT_PARENT = '1960b4693024a4439b1c9325e15131130cc1f60a'
COMMIT_NESTED = '0878c6880fa4fbadf0742751cf7b015f2d63a769'

REQUIRED_PKGS = [
    'openvino>=2025.0.0','nncf>=2.14.0',
    'torch>=2.1','torchvision','accelerate',
    'safetensors','numpy','pandas','matplotlib','tqdm','h5py',
    'onnx','onnxruntime','rich',
    'transformers>=4.45.2','tyro>=0.9.10','datasets==3.3.0','meshcat==0.3.2','logging_mp'
]

# Clone top-level repo if missing
if not REPO_DIR.exists():
    env = os.environ.copy(); env['GIT_LFS_SKIP_SMUDGE'] = '1'
    subprocess.check_call(['git','clone', REPO_URL, str(REPO_DIR)], env=env)
else:
    print('[INFO] Top-level repo exists.')

# Init / update submodules (ensure presence)
subprocess.check_call(['git','-C', str(REPO_DIR),'submodule','update','--init','--recursive'])

# Upgrade tooling + install pkgs
vrun(['-m','pip','install','-U','pip','setuptools','wheel'], 'Upgrade tooling')
vrun(['-m','pip','install','-U'] + REQUIRED_PKGS, 'Install required packages')

def is_git_dir(path: pathlib.Path):
    g = path / '.git'
    if g.is_dir():
        return True
    if g.is_file():
        # submodule pointer file
        return True
    return False

def ensure_commit(repo_path: pathlib.Path, commit: str, label: str):
    """
    Fetch commit into repo_path (handles submodule pointer .git file).
    """
    if not is_git_dir(repo_path):
        print(f'[WARN] {label} path {repo_path} is not a git repo.')
        return False
    # For submodule, .git is a file -> still use -C path
    # First try to see if commit exists
    has = subprocess.run(['git','-C',str(repo_path),'cat-file','-e',f'{commit}^{commit}'], capture_output=True)
    if has.returncode != 0:
        print(f'[INFO] Commit {commit[:8]} not present in {label}. Fetching all...')
        # remove shallow restrictions if any
        subprocess.run(['git','-C',str(repo_path),'fetch','--all','--tags','--prune'], check=True)
        # optional unshallow
        subprocess.run(['git','-C',str(repo_path),'fetch','--depth','1000000'], check=False)
        has2 = subprocess.run(['git','-C',str(repo_path),'cat-file','-e',commit], capture_output=True)
        if has2.returncode != 0:
            print(f'[ERROR] Commit {commit} still not found in {label}.')
            return False
    # Checkout in detached HEAD
    try:
        subprocess.check_call(['git','-C',str(repo_path),'checkout',commit])
        current = subprocess.check_output(['git','-C',str(repo_path),'rev-parse','HEAD']).decode().strip()
        if current != commit:
            print(f'[WARN] After checkout {label} HEAD={current[:8]} expected {commit[:8]}')
            return False
        print(f'[OK] {label} at commit {commit[:8]}')
        return True
    except subprocess.CalledProcessError as e:
        print(f'[ERROR] Checkout failed for {label}: {e}')
        return False

# Pin parent (inner repo)
parent_ok = ensure_commit(PARENT_DIR, COMMIT_PARENT, 'parent')

import os, subprocess
os.chdir('unitree_IL_lerobot/unitree_lerobot')
subprocess.check_call(['git','fetch','--all','--tags','--prune'])
subprocess.check_call(['git','checkout',COMMIT_PARENT])
print('HEAD:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
os.chdir('../../')

# Pin nested (if it is itself a git repo)
nested_ok = ensure_commit(NESTED_DIR, COMMIT_NESTED, 'nested')

# Patch Lerobot to let install with Python >= 3.10
import pathlib, re
PJP = pathlib.Path('unitree_IL_lerobot/pyproject.toml')
if PJP.exists():
    txt = PJP.read_text(encoding='utf-8')
    # Match a requires-python line and replace the value with ">=3.10"
    new_txt, n = re.subn(
        r'(?m)^(requires-python\s*=\s*")[^"]*(")\s*$',
        r'\1>=3.10\2',
        txt,
    )
    if n > 0:
        PJP.write_text(new_txt, encoding='utf-8')
        print(f'[PATCH] Updated requires-python in {PJP} to ">=3.10" (changed {n} occurrence).')
    else:
        print(f'[PATCH][INFO] No requires-python line changed in {PJP} (already relaxed or missing).')
    # Show the effective line for verification
    for line in new_txt.splitlines():
        if line.strip().startswith('requires-python'):
            print('[PATCH] Effective:', line)
            break
else:
    print('[PATCH][WARN] [pyproject.toml] not found at', PJP)

print(f'[STATUS] parent pinned={parent_ok} nested pinned={nested_ok}')

# Editable installs
vrun(['-m','pip','install','-e', str(REPO_DIR)], 'Editable install (top-level)')
if NESTED_DIR.exists():
    vrun(['-m','pip','install','-e', str(NESTED_DIR)], 'Editable install (nested lerobot)', check=True)
else:
    print('[ERROR] Missing nested path:', NESTED_DIR)

# Clean conflicting distributions
try:
    import importlib.metadata as md
    dist = md.distribution('lerobot')
    dist_path = pathlib.Path(dist.locate_file('lerobot')).resolve()
    if dist_path != NESTED_DIR.resolve() and NESTED_DIR.exists():
        print('[CLEANUP] Replacing existing lerobot distribution.')
        vrun(['-m','pip','uninstall','-y','lerobot'], 'Uninstall other lerobot', check=False)
        vrun(['-m','pip','install','-e', str(NESTED_DIR)], 'Reinstall target lerobot', check=True)
except Exception as e:
    print('[CLEANUP][INFO] Skip distribution check:', e)

# Prepend parent path
parent_str = str(PARENT_DIR.resolve())
if parent_str not in sys.path:
    sys.path.insert(0, parent_str)

# Import lerobot to verify
if 'lerobot' in sys.modules:
    del sys.modules['lerobot']
import lerobot
print('[IMPORT] lerobot ->', lerobot.__file__)

# Register kernel
vrun(['-m','pip','install','ipykernel'], 'Ensure ipykernel', check=True)

print('[DONE] Setup complete.')
print('[INFO] Requested parent commit :', COMMIT_PARENT)
print('[INFO] Requested nested commit :', COMMIT_NESTED)
try:
    parent_head = subprocess.check_output(['git','-C',str(PARENT_DIR),'rev-parse','HEAD']).decode().strip()
    print('[INFO] Actual parent HEAD      :', parent_head)
except Exception as e:
    print('[INFO] Parent HEAD unavailable:', e)
try:
    nested_head = subprocess.check_output(['git','-C',str(NESTED_DIR),'rev-parse','HEAD']).decode().strip()
    print('[INFO] Actual nested HEAD      :', nested_head)
except Exception as e:
    print('[INFO] Nested HEAD unavailable:', e)

The next cell configures all the paths. Before running it, select the Python 3 kernel. From the notebook menu, go to Kernel → Change kernel, then select Python 3 and click 'Select'.

In [ ]:
import sys; print(sys.executable)

# Configuration Parameters (Paths, Precision, Device)
import os, pathlib

CKPT_DIR = pathlib.Path('act_checkpoint') 
NOTEBOOK_DIR = pathlib.Path('.').resolve()
MODEL_DIR = pathlib.Path(os.getenv('ACT_PROJECT_ROOT', NOTEBOOK_DIR))
CHECKPOINT_PATH = pathlib.Path(os.getenv('ACT_CHECKPOINT', str(CKPT_DIR / 'model.safetensors')))
IR_OUTPUT_DIR = pathlib.Path(os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs'))
IR_OUTPUT_DIR.mkdir(exist_ok=True)
DATASET_ROOT = pathlib.Path(os.getenv('ACT_DATASET_ROOT', str(MODEL_DIR / 'dataset')))
STATS_PATH = pathlib.Path(os.getenv('ACT_STATS_PATH', str(CKPT_DIR / 'stats.json')))

PRECISIONS = ['FP32', 'FP16']
TARGET_DEVICE = os.getenv('ACT_TARGET_DEVICE', 'CPU')

print('Notebook directory:', NOTEBOOK_DIR)
print('Relative checkpoint dir:', CKPT_DIR)
print('Resolved checkpoint file path:', CHECKPOINT_PATH)
print('Dataset root:', DATASET_ROOT)
print('Stats path (may not exist yet):', STATS_PATH)
print('Output directory:', IR_OUTPUT_DIR)
print('Target device (OpenVINO):', TARGET_DEVICE)

## Acquire ACT Checkpoint Assets
Before running the next code cell, download the ACT model artifacts: `model.safetensors`, `config.json`, `stats.json`, and `train_config.json` into `act_checkpoint/`

In [ ]:
# Export environment variables
import os, pathlib
CKPT_DIR = pathlib.Path('act_checkpoint')
os.environ['ACT_CHECKPOINT'] = str(CKPT_DIR / 'model.safetensors')
os.environ['ACT_CONFIG_PATH'] = str(CKPT_DIR / 'config.json')
os.environ['ACT_TRAIN_CONFIG_PATH'] = str(CKPT_DIR / 'train_config.json')
stats_path = CKPT_DIR / 'stats.json'
if stats_path.exists():
    os.environ['ACT_STATS_PATH'] = str(stats_path)
print('[INFO] Checkpoint directory (relative):', CKPT_DIR)
print('[INFO] Environment variables:')
for k in ['ACT_CHECKPOINT','ACT_CONFIG_PATH','ACT_TRAIN_CONFIG_PATH','ACT_STATS_PATH']:
    if k in os.environ:
        print('  ', k, '=', os.environ[k])


# Load ACT Policy (Overview)

Next code cell does the followings:
- Verifies both model.safetensors and config.json file exist; aborts with clear errors if missing.
- Parses config.json and filters keys to ACTConfig’s constructor.
- Wraps feature definitions into PolicyFeature and normalization mapping into NormalizationMode.
- Instantiates ACTConfig, builds ACTPolicy, loads weights (strict=False), switches to eval().
- Extracts action dimension, chunk_size (default 100 if absent), and discovers camera feature keys (observation.images.*).
- Prints parameter count and detected cameras for later conversion steps.

In [ ]:
# Load Original ACT Model
import os, json, inspect, pathlib, sys, importlib
from safetensors.torch import load_file
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.configs.types import PolicyFeature, FeatureType, NormalizationMode

CHECKPOINT_PATH = pathlib.Path(os.getenv('ACT_CHECKPOINT', 'act_checkpoint/model.safetensors'))
CONFIG_PATH = pathlib.Path(os.getenv('ACT_CONFIG_PATH', str(CHECKPOINT_PATH.parent / 'config.json')))

print('[LOAD] CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('[LOAD] CONFIG_PATH     =', CONFIG_PATH)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint file not found at {CHECKPOINT_PATH}.\n"
        "Ensure you have: (1) placed model.safetensors in act_checkpoint/, or (2) set ACT_CHECKPOINT env var, then re-run this cell."
    )
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"config.json not found at {CONFIG_PATH}.\n"
        "Place config.json next to the checkpoint (act_checkpoint/config.json) or set ACT_CONFIG_PATH."
    )

with open(CONFIG_PATH, 'r') as f:
    cfg_dict = json.load(f)

# Filter config keys to ACTConfig signature
valid_keys = set(inspect.signature(ACTConfig.__init__).parameters.keys()); valid_keys.discard('self')
filtered_cfg = {k: v for k, v in cfg_dict.items() if k in valid_keys}

# Helper wrappers
def wrap_features(feat_dict):
    return {k: PolicyFeature(type=FeatureType(v['type']), shape=tuple(v['shape'])) for k, v in feat_dict.items()}

def wrap_norm_map(norm_map):
    return {FeatureType(k): NormalizationMode(v) for k, v in norm_map.items()}

if 'input_features' in filtered_cfg:
    filtered_cfg['input_features'] = wrap_features(filtered_cfg['input_features'])
if 'output_features' in filtered_cfg:
    filtered_cfg['output_features'] = wrap_features(filtered_cfg['output_features'])
if 'normalization_mapping' in filtered_cfg:
    filtered_cfg['normalization_mapping'] = wrap_norm_map(filtered_cfg['normalization_mapping'])

act_config = ACTConfig(**filtered_cfg)
act_config.use_vae = False
policy = ACTPolicy(act_config)
weights = load_file(str(CHECKPOINT_PATH))
policy.load_state_dict(weights, strict=False)
policy.eval()
print('Loaded ACTPolicy from safetensors. Params:', sum(p.numel() for p in policy.parameters()))

# Extract dimensions
action_dim = filtered_cfg['output_features']['action'].shape[0]
chunk_size = filtered_cfg.get('chunk_size', 100)
# Camera keys
camera_keys = sorted([k for k in cfg_dict['input_features'] if k.startswith('observation.images.')])
print('Detected cameras:', camera_keys)


## Inspect model and build dummy inputs (for conversion)

The next code cell:
- Reads input feature dimensions from `policy.config` (state, per-camera images, environment_state).
- Allocates zero tensors with batch size 1 for:
  - `observation.state` [1, state_dim]
  - each camera image [1, C, H, W] using shapes from the config
  - `action_is_pad` [1, chunk_size] (bool)
  - `action` sequence [1, chunk_size, action_dim]
  - optional `observation.environment_state` [1, env_dim]
- Prints all shapes to verify setup. These tensors will be used for tracing/export in the next steps.

In [ ]:
# Inspect Model Architecture and Construct Full Dummy Inputs
import torch 

state_dim = policy.config.input_features['observation.state'].shape[0]
chunk_size = chunk_size  # from previous cell
H, W = 480, 640
cams = camera_keys

# Use shapes from config if specified
image_tensors = []
for cam in cams:
    shape = policy.config.input_features[cam].shape  # e.g. [3, H, W]
    img = torch.zeros(1, *shape, dtype=torch.float32)
    image_tensors.append(img)

state = torch.zeros(1, state_dim, dtype=torch.float32)
action_is_pad = torch.zeros(1, chunk_size, dtype=torch.bool)
action_seq = torch.zeros(1, chunk_size, action_dim, dtype=torch.float32)

env_state = None
if 'observation.environment_state' in policy.config.input_features:
    env_dim = policy.config.input_features['observation.environment_state'].shape[0]
    env_state = torch.zeros(1, env_dim, dtype=torch.float32)

print('State shape:', state.shape)
print('Image shapes:', [t.shape for t in image_tensors])
print('Action pad shape:', action_is_pad.shape)
print('Action seq shape:', action_seq.shape)
if env_state is not None:
    print('Environment state shape:', env_state.shape)


In [ ]:
# Prepare Ordered Inputs
# Order: observation.state, each camera image, action_is_pad, action, optional environment_state
ordered_inputs = [state] + image_tensors + [action_is_pad, action_seq] + ([env_state] if env_state is not None else [])
print('Ordered input tensor shapes:', [t.shape for t in ordered_inputs])


## Direct PyTorch to OpenVINO IR (No ONNX)

Convert the loaded ACT policy directly from PyTorch to OpenVINO IR using the FX frontend. This bypasses ONNX export.

The next cell:
- Wraps ACTPolicy in a DirectWrapper; internal action_is_pad/action are synthesized (not IR inputs).
- Converts with ov.convert_model using example inputs; includes env if present.
- Renames IR inputs: observation_state, observation_images_0..N, optional observation_environment_state.
- Validates input count and prints port names and partial shapes.
- Saves FP32 IR (act_model_direct_fp32.xml/bin). 
- To produce FP16, call ov.save_model(..., compress_to_fp16=True).


In [ ]:
# Direct PyTorch to OpenVINO IR (FP32 by default, FP16 instructions included)
import torch, pathlib, openvino as ov

# Required objects from previous cells.
required = ['policy', 'camera_keys', 'state', 'image_tensors', 'action_dim', 'chunk_size']
for sym in required:
    if sym not in globals():
        raise RuntimeError(f'Missing `{sym}`. Run earlier cells first.')

env_present = 'env_state' in globals() and env_state is not None

class DirectWrapper(torch.nn.Module):
    """Expose only observation_state, per-camera images, optional env.
    Internal temporal tensors (action_is_pad, action) are synthesized so they do NOT become IR inputs.
    This keeps the IR minimal and matches evaluation expectations.
    """
    def __init__(self, act_policy, camera_keys, chunk_size, action_dim, env_present=False):
        super().__init__()
        self.model = act_policy
        self.camera_keys = camera_keys
        self.chunk_size = chunk_size
        self.action_dim = action_dim
        self.env_present = env_present
    def forward(self, observation_state, *cams_and_env):
        num_cams = len(self.camera_keys)
        cam_tensors = cams_and_env[:num_cams]
        env_tensor = cams_and_env[num_cams] if self.env_present and len(cams_and_env) > num_cams else None
        B = observation_state.shape[0]
        device = observation_state.device
        action_is_pad_local = torch.zeros(B, self.chunk_size, dtype=torch.bool, device=device)
        action_local = torch.zeros(B, self.chunk_size, self.action_dim, dtype=torch.float32, device=device)
        batch = {
            'observation.state': observation_state,
            'action_is_pad': action_is_pad_local,
            'action': action_local,
            'observation.images': list(cam_tensors)
        }
        for i, key in enumerate(self.camera_keys):
            batch[key] = cam_tensors[i]
        if env_tensor is not None:
            batch['observation.environment_state'] = env_tensor
        out = self.model.model(batch)
        if isinstance(out, tuple):
            out = out[0]
        return out

# Construct example inputs for tracing
example_inputs = [torch.randn_like(state)] + [torch.randn_like(t) for t in image_tensors]
if env_present:
    example_inputs.append(torch.randn_like(env_state))

wrapper = DirectWrapper(policy, camera_keys, chunk_size, action_dim, env_present).eval()
print('[DIRECT] Converting via ov.convert_model (FX)...')
ov_model = ov.convert_model(wrapper, example_input=tuple(example_inputs))

# Rename ports to evaluation expectations
inputs = ov_model.inputs
expected = 1 + len(camera_keys) + (1 if env_present else 0)
if len(inputs) != expected:
    raise RuntimeError(f'Unexpected IR input count {len(inputs)} vs expected {expected}.')

def _set_names(inp, desired: str):
    node = inp.get_node()
    try:
        node.set_friendly_name(desired)
    except Exception:
        pass
    try:
        inp.get_tensor().set_names({desired})
    except Exception as e:
        print('[WARN] Failed to set tensor name for', desired, ':', e)

_set_names(inputs[0], 'observation_state')
for i in range(len(camera_keys)):
    _set_names(inputs[i+1], f'observation_images_{i}')
if env_present:
    _set_names(inputs[-1], 'observation_environment_state')

print('[DIRECT] Final IR input ports (friendly_name / tensor names / partial shape):')
dynamic_present = False
for inp in ov_model.inputs:
    node = inp.get_node()
    try:
        tnames = list(inp.get_tensor().get_names())
    except Exception:
        tnames = []
    try:
        ps = inp.get_partial_shape()
    except Exception:
        ps = None
    if ps is not None and ps.is_static:
        try:
            concrete = ps.to_shape()
            shape_repr = '[' + ', '.join(str(d) for d in concrete) + ']'
        except Exception:
            shape_repr = str(ps)
    else:
        dynamic_present = True
        shape_repr = str(ps) if ps is not None else 'Unknown(dynamic)'
    print(f"  - {tnames[0] if tnames else node.get_friendly_name()} | tensor_names={tnames} | partial_shape={shape_repr}")

IR_OUTPUT_DIR = pathlib.Path(globals().get('IR_OUTPUT_DIR', 'openvino_ir_outputs'))
IR_OUTPUT_DIR.mkdir(exist_ok=True)

# Save FP32 IR
xml_fp32 = IR_OUTPUT_DIR / 'act_model_direct_fp32.xml'
ov.save_model(ov_model, str(xml_fp32))
print('[DIRECT] Saved FP32 XML:', xml_fp32, '| size:', xml_fp32.stat().st_size if xml_fp32.exists() else 0)
print('[DIRECT] Saved FP32 BIN :', xml_fp32.with_suffix('.bin'), '| size:', xml_fp32.with_suffix('.bin').stat().st_size if xml_fp32.with_suffix('.bin').exists() else 0)

# --- FP16 Guidance ---
# To also emit an FP16 version (weights compressed to half precision) uncomment:
# xml_fp16 = IR_OUTPUT_DIR / 'act_model_direct_fp16.xml'
# ov.save_model(ov_model, str(xml_fp16), compress_to_fp16=True)
# print('[DIRECT] Saved FP16 XML:', xml_fp16)
# print('[DIRECT] Saved FP16 BIN :', xml_fp16.with_suffix('.bin'))

print('\n[HINT] In evaluation build input dict using: observation_state, observation_images_0..N, (optional) observation_environment_state.')
print('[DONE] Direct conversion complete. (See comments above for FP16 save).')


## Optional: INT8 Quantization (Post-Training)
This section generates an INT8 (quantized) OpenVINO model using the helper script `quantize_int8_helper.py` found in this folder.

The helper runs with the following arguments:
  - `--model_xml` FP32 IR path
  - `--stats_path` training stats
  - `--dataset_root` calibration dataset
  - `--output_dir` `openvino_ir_outputs/int8`
  - `--num_calib_samples` (default 300)
  - `--preset` (`performance` or `accuracy`)
  
Why INT8?
- Smaller binary size.
- Potential throughput / latency gains (depends on CPU / GPU / VPU).
- Usually minimal accuracy drop if calibration data is representative.

What you need first:
1. A FP32 IR (e.g. `act_model_direct_fp32.xml` created above).
2. `stats.json` from training (already exported earlier or placed into `act_checkpoint/`).
3. A local LeRobot dataset root with episode data (env var `ACT_DATASET_ROOT` or edit path below).
4. Packages: `openvino-dev` and `nncf` installed.

Calibration parameters:
- `num_calib_samples`: how many sequential steps to sample (default 300). Increase if quality degrades.
- `preset`: `performance` (aggressive compression) or `accuracy` (more conservative).

Outputs:
- `int8/model_int8.xml` and `int8/model_int8.bin` in the IR output directory.

Note:
- Typical runtime: ~2–15 minutes for 300 samples on CPU; faster on a modern GPU.

In [ ]:
# Uses quantize_int8_helper.py to produce an INT8 model from the FP32 IR.
# Relies on variables defined earlier: IR_OUTPUT_DIR, STATS_PATH, DATASET_ROOT.
# If the kernel was reset and those are missing, it falls back to env vars or defaults.
# If dataset is missing, prints guidance and shows how to set an override.
import sys, runpy, pathlib, os
from datetime import datetime

IR_OUTPUT_DIR = pathlib.Path(globals().get('IR_OUTPUT_DIR', os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs')))
STATS_JSON = pathlib.Path(globals().get('STATS_PATH', os.getenv('ACT_STATS_PATH', 'act_checkpoint/stats.json')))
DATASET_ROOT = pathlib.Path('dataset/G1_BlockStacking_Dataset')
FP32_XML = IR_OUTPUT_DIR / 'act_model_direct_fp32.xml'
OUT_INT8_DIR = IR_OUTPUT_DIR / 'int8'
CALIB_SAMPLES = 300  # Tune if needed (increase for potentially better accuracy)
PRESET = 'performance'  # or 'accuracy'
SCRIPT_PATH = pathlib.Path('quantize_int8_helper.py')  # Expected in same directory

print('[INT8] Resolved paths:')
print('  IR_OUTPUT_DIR        =', IR_OUTPUT_DIR)
print('  FP32_XML             =', FP32_XML)
print('  STATS_JSON           =', STATS_JSON)
print('  DATASET_ROOT         =', DATASET_ROOT)
print('  SCRIPT_PATH          =', SCRIPT_PATH)
print('  OUT_INT8_DIR         =', OUT_INT8_DIR)
print('  CALIB_SAMPLES        =', CALIB_SAMPLES)
print('  PRESET               =', PRESET)

missing_msgs = []
if not FP32_XML.exists():
    missing_msgs.append(f'FP32 IR not found at {FP32_XML}. Run the direct conversion cell first.')
if not STATS_JSON.exists():
    missing_msgs.append(f'stats.json not found at {STATS_JSON}. Provide training stats for normalization.')
if not DATASET_ROOT.exists():
    missing_msgs.append(f'Dataset root not found at {DATASET_ROOT}. Set ACT_DATASET_ROOT or provide a valid path.')

if not SCRIPT_PATH.exists():
    missing_msgs.append(f'quantize_int8_helper.py not found at {SCRIPT_PATH}. Place the helper script alongside the notebook.')
if missing_msgs:
    raise FileNotFoundError('\n'.join(missing_msgs))

OUT_INT8_DIR.mkdir(exist_ok=True)
print(f'\n[INT8] Starting quantization at {datetime.utcnow().isoformat()}Z')
print(f'[INT8] Using FP32 model: {FP32_XML.name}')
print(f'[INT8] Stats file       : {STATS_JSON.name}')
print(f'[INT8] Dataset root     : {DATASET_ROOT}')
print(f'[INT8] Output directory : {OUT_INT8_DIR}')
print(f'[INT8] Calibration samples={CALIB_SAMPLES} preset={PRESET}')

argv_backup = sys.argv
sys.argv = [
    'quantize_int8_helper.py',
    '--model_xml', str(FP32_XML),
    '--stats_path', str(STATS_JSON),
    '--dataset_root', str(DATASET_ROOT),
    '--output_dir', str(OUT_INT8_DIR),
    '--num_calib_samples', str(CALIB_SAMPLES),
    '--preset', PRESET
]
print('[INT8] Running helper script with args:\n ', ' '.join(sys.argv))
try:
    runpy.run_path(str(SCRIPT_PATH), run_name='__main__')
finally:
    sys.argv = argv_backup

INT8_XML = OUT_INT8_DIR / 'model_int8.xml'
if INT8_XML.exists():
    print('[INT8] Success. INT8 model at', INT8_XML)
    print('[INT8] File sizes: XML', INT8_XML.stat().st_size, 'BIN', INT8_XML.with_suffix('.bin').stat().st_size)
else:
    print('[INT8] Quantization finished but INT8 artifact missing. Check logs above for errors.')


## Evaluation & Comparison Plotting

Next cell runs evaluation and comparison for each OpenVINO IR model variant (FP32, INT8) using the helper script. It generates action comparison plots for each variant, comparing OpenVINO outputs to the baseline PyTorch model. Results are saved as PNG figures for further analysis.


In [ ]:
# Evaluation & Comparison Plotting
import sys, os, pathlib, datetime, runpy, shutil, traceback

REQUIRED_EVAL_PKGS = ["openvino", "torch", "numpy", "matplotlib"]
for mod in REQUIRED_EVAL_PKGS:
    try:
        __import__(mod)
    except Exception as e:
        print(f"[EVAL][WARN] Missing module '{mod}' ({e}).")

needed_syms = ["IR_OUTPUT_DIR", "CHECKPOINT_PATH", "STATS_PATH", "TARGET_DEVICE"]
for sym in needed_syms:
    if sym not in globals():
        raise RuntimeError(f"[EVAL] Missing `{sym}`; rerun earlier cells.")

EVAL_SCRIPT = pathlib.Path("eval_openvino_model_helper.py")
if not EVAL_SCRIPT.exists():
    raise FileNotFoundError(f"Helper script missing: {EVAL_SCRIPT}")

stats_path = pathlib.Path(STATS_PATH)
DATASET_ROOT = pathlib.Path("dataset/G1_BlockStacking_Dataset")
if not stats_path.exists():
    fallback = DATASET_ROOT / "meta" / "stats.json"
    if fallback.exists():
        stats_path = fallback
        print(f"[EVAL] Using fallback stats path: {stats_path}")
    else:
        raise FileNotFoundError(f"stats.json not found at {STATS_PATH} or {fallback}")

MODEL_VARIANTS = [
    ("direct_fp32", IR_OUTPUT_DIR / "act_model_direct_fp32.xml"),
    ("direct_fp16", IR_OUTPUT_DIR / "act_model_direct_fp16.xml"),
    ("mo_fp32",     IR_OUTPUT_DIR / "act_model_fp32.xml"),
    ("int8",        IR_OUTPUT_DIR / "int8" / "model_int8.xml"),
]
MODEL_VARIANTS = [(lbl, p) for lbl, p in MODEL_VARIANTS if p.exists()]
if not MODEL_VARIANTS:
    raise RuntimeError("[EVAL] No model variants found.")

print('[EVAL] Variants discovered:', ', '.join(lbl for lbl, _ in MODEL_VARIANTS))
print('[EVAL] Stats path         :', stats_path)
print('[EVAL] Dataset root       :', DATASET_ROOT)
print('[EVAL] Policy directory   :', CHECKPOINT_PATH.parent)
print('[EVAL] Device             :', TARGET_DEVICE)

ENV_KEYS = ["OPENVINO_MODEL_PATH", "STATS_PATH", "OPENVINO_PRECISION_HINT"]
original_env = {k: os.environ.get(k) for k in ENV_KEYS}
figures = []

def infer_precision(label: str, path: pathlib.Path) -> str:
    ll = label.lower(); fp = str(path).lower()
    if "int8" in ll or "int8" in fp: return "INT8"
    if "fp16" in ll or "fp16" in fp: return "FP16"
    return "FP32"

# Episodes logic: if dataset exists use 1, else 0 (synthetic path expected in helper)
episodes = 1 if DATASET_ROOT.exists() else 0
if episodes == 0:
    print("[EVAL][INFO] Dataset root missing; running with episodes=0 (synthetic / may limit evaluation).")

for label, model_xml in MODEL_VARIANTS:
    precision_hint = infer_precision(label, model_xml)
    print(f"\n[EVAL] Variant '{label}' -> {model_xml.name} (precision={precision_hint}, device={TARGET_DEVICE})")

    os.environ['OPENVINO_MODEL_PATH'] = str(model_xml)
    os.environ['STATS_PATH'] = str(stats_path)
    os.environ['OPENVINO_PRECISION_HINT'] = precision_hint

    # Compatibility shim for legacy lerobot missing PolicyAction
    try:
        from lerobot.processor import PolicyAction  # noqa
    except Exception:
        try:
            import lerobot.processor as proc
            class PolicyAction:  # stub
                pass
            proc.PolicyAction = PolicyAction
            print("[EVAL][SHIM] Injected PolicyAction stub.")
        except Exception as e:
            print("[EVAL][SHIM][FAIL] Could not inject PolicyAction stub:", e)

    argv_backup = sys.argv
    sys.argv = [
        'eval_openvino_model_helper.py',
        '--repo_id=None',
        f'--root={DATASET_ROOT}',
        f'--policy.path={CHECKPOINT_PATH.parent}',
        '--policy.device=cpu',          # force CPU to avoid CUDA mismatch
        f'--episodes={episodes}',
        '--visualization=False',
        '--use_dataset=False' if episodes == 0 else '--use_dataset=True',
    ]
    print('[EVAL] sys.argv ->', ' '.join(sys.argv))
    try:
        runpy.run_path(str(EVAL_SCRIPT), run_name='__main__')
    except SystemExit as e:
        print(f"[EVAL][ERROR] SystemExit({e.code}) for {label}.")
    except Exception as e:
        print(f"[EVAL][ERROR] Exception during evaluation of {label}: {e}")
        traceback.print_exc()
    finally:
        sys.argv = argv_backup
        for k, v in original_env.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

    fig_src = pathlib.Path('actions_comparison.png')
    if fig_src.exists():
        timestamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
        fig_dst = pathlib.Path(f'{fig_src.stem}_{label}.png')
        if fig_dst.exists():
            fig_dst = pathlib.Path(f'{fig_src.stem}_{label}_{timestamp}.png')
        shutil.move(str(fig_src), str(fig_dst))
        figures.append(fig_dst)
        print('[EVAL] Saved figure ->', fig_dst)
    else:
        print('[EVAL][WARN] No figure produced for', label)

print('\n[EVAL] Summary of figures:')
for f in figures:
    print('  -', f)
if not figures:
    print('[EVAL] No figures generated.')
print('[EVAL] Done.')